In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# 엑셀 파일 경로 지정
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/클러스터링 데이터_가중치 최종본.xlsx")
# 컬럼명 확인 (예: 위도, 경도라는 한글 컬럼이 있는 경우)


df = df.rename(columns={'위도': 'lat', '경도': 'lon','가중치':'weight'})    #위경도 이름 바꾸기
print(df.columns)

Index(['장소명', '주소', 'lat', 'lon', '만3세유아수', '만4세유아수', '만5세유아수', '혼합유아수',
       '특수유아수', '정원', '현원', '시설종류명(시설유형)', '시설종류상세명(시설종류)', '주소(읍면동)',
       '법정동 아동수(0~9세)', '법정동 세대수', 'k-전체세대수', 'weight', '1학년수', '2학년수',
       '3학년수'],
      dtype='object')


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from math import radians, sin, cos, sqrt, atan2
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

# Haversine 거리 계산 (미터)
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = radians(lat1), radians(lat2)
    d_phi = radians(lat2 - lat1)
    d_lambda = radians(lon2 - lon1)
    a = sin(d_phi/2)**2 + cos(phi1)*cos(phi2)*sin(d_lambda/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

def kmeans_clustering_with_weights(df, k_max):
    summary_table = []
    base_map = folium.Map(location=[df['lat'].mean(), df['lon'].mean()], zoom_start=14)
    marker_cluster = MarkerCluster().add_to(base_map)

    max_k = min(len(df), k_max)
    for k in range(1, max_k+1):
        coords = df[['lat','lon']]
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        df['cluster'] = kmeans.fit_predict(coords, sample_weight=df['weight'])
        centers = kmeans.cluster_centers_

        for cid in range(k):
            cluster_data = df[df['cluster']==cid]
            center_lat, center_lon = centers[cid][0], centers[cid][1]

            # 각 점까지 거리 계산
            distances = cluster_data.apply(
                lambda r: haversine_distance(center_lat, center_lon, r['lat'], r['lon']),
                axis=1
            )
            # 개수 계산
            within_mask = distances <= 540
            count_within = within_mask.sum()
            count_over   = (~within_mask).sum()
            max_dist     = distances.max()

            summary_table.append({
                'k': k,
                'cluster': chr(65+cid),
                'num_points': len(cluster_data),
                'count_within_540m': int(count_within),
                'count_over_540m':   int(count_over),
                'max_distance_m': round(float(max_dist),2)
            })

            # 지도에 중심점 추가
            folium.Marker(
                location=[center_lat, center_lon],
                popup=f"k={k}, Cluster {chr(65+cid)}",
                icon=folium.Icon(color='red', icon='star')
            ).add_to(marker_cluster)

    summary_df = pd.DataFrame(summary_table)

    # k별 540m 이내/초과 군집 수 요약
    # (한 군집의 max_distance 기준)
    k_grouped = summary_df.groupby('k').agg(
        num_clusters=('cluster','count'),
        clusters_within_540m=('count_over_540m', lambda x: (x==0).sum()),
        clusters_over_540m=('count_over_540m', lambda x: (x>0).sum())
    ).reset_index()

    return base_map, summary_df, k_grouped

# — 예시 실행 —
# df는 lon, lat, weight 컬럼을 가진 DataFrame이어야 합니다.
# df = pd.read_csv("your_data.csv") 등으로 로드하세요.

# 최대 k를 30으로 제한
base_map, summary_df, k_grouped = kmeans_clustering_with_weights(df, k_max=60)

# 결과 출력
display(summary_df)    # (k, cluster)별 상세 정보
display(k_grouped)     # k별 540m 이내/초과 군집 수 요약
base_map               # 지도 시각화


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# summary_df 저장 및 다운로드
summary_df.to_excel("cluster_summary_details.xlsx", index=False)

from google.colab import files
files.download("cluster_summary_details.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# k_grouped 저장 및 다운로드
k_grouped.to_excel("cluster_summary_by_k.xlsx", index=False)

from google.colab import files
files.download("cluster_summary_by_k.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from scipy.spatial import ConvexHull
import folium
from folium import features

def visualize_clusters_with_hulls(df, k_value):
    coords = df[['lat', 'lon']]
    kmeans = KMeans(n_clusters=k_value, random_state=42, n_init='auto')
    df['cluster'] = kmeans.fit_predict(coords, sample_weight=df['weight'])
    centers = kmeans.cluster_centers_

    # 지도 설정
    m = folium.Map(location=[df['lat'].mean(), df['lon'].mean()], zoom_start=14)

    # 색상 리스트
    colors = [
        "red", "blue", "green", "purple", "orange", "darkred", "darkblue",
        "darkgreen", "cadetblue", "darkpurple", "black", "pink", "lightblue",
        "lightgreen", "gray", "brown", "olive", "teal", "navy", "maroon",
        "gold", "cyan", "magenta", "lime", "chocolate", "crimson", "indigo",
        "plum", "salmon", "khaki"
    ]

    for cluster_id in range(k_value):
        cluster_data = df[df['cluster'] == cluster_id]
        points = cluster_data[['lat', 'lon']].values

        if len(points) >= 3:
            # ConvexHull 계산
            hull = ConvexHull(points)
            hull_points = [points[i] for i in hull.vertices]

            # 폴리곤 그리기
            folium.Polygon(
                locations=hull_points,
                color=colors[cluster_id % len(colors)],
                fill=True,
                fill_opacity=0.2,
                weight=2,
                popup=f'Cluster {chr(65 + cluster_id)}'
            ).add_to(m)

        # 각 점 찍기
        for _, row in cluster_data.iterrows():
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=4,
                color='black',
                weight=1,
                fill=True,
                fill_color=colors[cluster_id % len(colors)],
                fill_opacity=0.8
            ).add_to(m)

        # 중심점 표시
        center = centers[cluster_id]
        folium.Marker(
            location=[center[0], center[1]],
            popup=f"Center {chr(65 + cluster_id)}",
            icon=folium.Icon(color='red', icon='star')
        ).add_to(m)

    return m

# k=26일 때 실행
k26_hull_map = visualize_clusters_with_hulls(df, k_value=26)
k26_hull_map


In [ ]:
from sklearn.cluster import KMeans

def get_cluster_centers(df, k_value):
    coords = df[['lat', 'lon']]
    kmeans = KMeans(n_clusters=k_value, random_state=42, n_init='auto')
    df['cluster'] = kmeans.fit_predict(coords, sample_weight=df['weight'])
    centers = kmeans.cluster_centers_

    # 클러스터명과 중심 좌표를 담은 DataFrame 만들기
    center_df = pd.DataFrame({
        'cluster': [chr(65 + i) for i in range(k_value)],
        'center_lat': [center[0] for center in centers],
        'center_lon': [center[1] for center in centers]
    })

    return center_df

# 실행
center_df = get_cluster_centers(df, k_value=26)
display(center_df)


,cluster,center_lat,center_lon
0,A,37.569994,127.080974
1,B,37.543322,127.089651
2,C,37.536619,127.079574
3,D,37.543443,127.067402
4,E,37.553293,127.083153
5,F,37.543484,127.103145
6,G,37.551759,127.072408
7,H,37.543988,127.097780
8,I,37.530394,127.083035
9,J,37.557197,127.091500


In [ ]:
k26_hull_map.save("k26_cluster_hulls.html")
from google.colab import files
files.download("k26_cluster_hulls.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 엑셀 파일로 저장
center_df.to_excel("cluster_centers_k26.xlsx", index=False)
from google.colab import files
center_df.to_excel("cluster_centers_k26.xlsx", index=False)
files.download("cluster_centers_k26.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>